# ECG-FM Stage 2 — Kaggle One-Click

### Before running:

**1. Add datasets** (Edit → Settings → Add-ons):
- `shlomihazan/ptbxl-500hz` — PTB-XL 500 Hz signals
- `shlomihazan/ecgfm-models` — model weights + training scripts

**2. Set GPU to T4:** Edit → Settings → Accelerator → **GPU T4 x2**
- NOTE: P100 is NOT compatible with the current PyTorch — T4 is required.

**3. Run the single cell below.**

---
Output `ecgfm_stage2.pt` will appear in the Output tab when done (~3 hrs).

In [ ]:
import os, shutil, zipfile, torch, psutil, pandas as pd
from pathlib import Path

# ── 1. Check GPU ──────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError('No GPU. Edit -> Settings -> Accelerator -> GPU T4 x2')

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
print(f'GPU : {gpu_name}  (sm_{major}{minor})')
print(f'VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)} GB')
if major < 7:
    raise RuntimeError(f'{gpu_name} (sm_{major}{minor}) NOT compatible. Need T4.')
print('GPU OK\n')

# ── 2. Find datasets ──────────────────────────────────────────────────────────
MODEL_PATH = None
for f in Path('/kaggle/input').rglob('ecgfm_stage1.pt'):
    MODEL_PATH = str(f.parent); break

PTBXL_PATH = None
for f in Path('/kaggle/input').rglob('records500.zip'):
    PTBXL_PATH = str(f.parent); break
if not PTBXL_PATH:
    for d in Path('/kaggle/input').rglob('records500'):
        if d.is_dir(): PTBXL_PATH = str(d.parent); break

if not MODEL_PATH:
    raise FileNotFoundError('ecgfm-models not found. Add shlomihazan/ecgfm-models')
if not PTBXL_PATH:
    raise FileNotFoundError('ptbxl-500hz not found. Add shlomihazan/ptbxl-500hz')

for fname in ['ecgfm_finetune.py', 'ecgfm_encoder.py', 'cnn_classifier.py',
              'mimic_iv_ecg_physionet_pretrained.pt']:
    if not os.path.exists(f'{MODEL_PATH}/{fname}'):
        raise FileNotFoundError(f'Missing {fname} in ecgfm-models')

print(f'PTB-XL path : {PTBXL_PATH}')
print(f'Model path  : {MODEL_PATH}\n')

# ── 3. Set up working directory ───────────────────────────────────────────────
WORKDIR = '/kaggle/working/ecgfm'
os.makedirs(f'{WORKDIR}/models/ecgfm', exist_ok=True)
os.makedirs(f'{WORKDIR}/ekg_datasets', exist_ok=True)

for script in ['ecgfm_finetune.py', 'ecgfm_encoder.py', 'cnn_classifier.py']:
    shutil.copy(f'{MODEL_PATH}/{script}', f'{WORKDIR}/{script}')

shutil.copy(f'{MODEL_PATH}/mimic_iv_ecg_physionet_pretrained.pt',
            f'{WORKDIR}/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')
shutil.copy(f'{MODEL_PATH}/ecgfm_stage1.pt', f'{WORKDIR}/models/ecgfm_stage1.pt')

# ── 4. Extract PTB-XL ────────────────────────────────────────────────────────
ptbxl_dest = f'{WORKDIR}/ekg_datasets/ptbxl'
if os.path.islink(ptbxl_dest): os.remove(ptbxl_dest)

zip_path = f'{PTBXL_PATH}/records500.zip'
if os.path.exists(zip_path):
    os.makedirs(ptbxl_dest, exist_ok=True)
    print('Extracting records500.zip (~2 min)...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(ptbxl_dest)
    shutil.copy(f'{PTBXL_PATH}/ptbxl_database.csv', f'{ptbxl_dest}/ptbxl_database.csv')
    shutil.copy(f'{PTBXL_PATH}/scp_statements.csv',  f'{ptbxl_dest}/scp_statements.csv')
else:
    os.symlink(os.path.abspath(PTBXL_PATH), ptbxl_dest)

# ── 5. Diagnose path alignment ────────────────────────────────────────────────
dat_files = list(Path(ptbxl_dest).rglob('*.dat'))
print(f'Extracted .dat files : {len(dat_files)}')
print(f'First 3 extracted paths (relative to ptbxl_dest):')
for f in sorted(dat_files)[:3]:
    print(f'  {f.relative_to(ptbxl_dest)}')

csv_path = f'{ptbxl_dest}/ptbxl_database.csv'
meta = pd.read_csv(csv_path, index_col='ecg_id')
print(f'\nCSV filename_hr samples:')
for v in meta['filename_hr'].head(3):
    print(f'  {v}')

# Check if first CSV path resolves correctly
sample_hr = meta['filename_hr'].iloc[0]
sample_full = f'{ptbxl_dest}/{sample_hr}.dat'
print(f'\nExpected .dat path : {sample_full}')
print(f'File exists        : {os.path.exists(sample_full)}')

# Auto-fix: if CSV path has extra prefix, strip it
if not os.path.exists(sample_full):
    # Try without leading 'records500/' prefix
    stripped = str(Path(sample_hr).parts[-2] + '/' + Path(sample_hr).name)
    alt = f'{ptbxl_dest}/{stripped}.dat'
    print(f'Trying stripped path: {alt}  exists={os.path.exists(alt)}')
    # Find actual relative path of first dat file
    first_dat = sorted(dat_files)[0]
    first_rel = str(first_dat.relative_to(ptbxl_dest).with_suffix(''))
    print(f'Actual first file   : {first_rel}')

print()
if not os.path.exists(sample_full):
    raise RuntimeError(
        f'Path mismatch! CSV expects "{sample_hr}" but files are at "{sorted(dat_files)[0].relative_to(ptbxl_dest)}".\n'
        'Check output above to diagnose.'
    )

n_dat = len(dat_files)
print(f'PTB-XL signal files: {n_dat}/21837')
if n_dat < 20000:
    raise RuntimeError(f'Only {n_dat} .dat files — dataset incomplete.')

print(f'Encoder : {round(os.path.getsize(WORKDIR+"/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt")/1e6)} MB')
print(f'Stage1  : {round(os.path.getsize(WORKDIR+"/models/ecgfm_stage1.pt")/1e6)} MB')
print('Setup complete\n')

# ── 6. Install wfdb from bundled wheel ────────────────────────────────────────
import subprocess, sys
whl = f'{MODEL_PATH}/wfdb-4.3.1-py3-none-any.whl'
if os.path.exists(whl):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', whl], check=True)
    print('wfdb installed from local wheel\n')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
    print('wfdb installed from PyPI\n')
print('Dependencies ready\n')

# ── 7. Train ──────────────────────────────────────────────────────────────────
os.chdir(WORKDIR)
!python -u ecgfm_finetune.py --stage 2 --batch_s2 8

# ── 8. Results ────────────────────────────────────────────────────────────────
model_path = f'{WORKDIR}/models/ecgfm_stage2.pt'
if os.path.exists(model_path):
    print(f'\necgfm_stage2.pt saved ({round(os.path.getsize(model_path)/1e6)} MB)')
    ckpt = torch.load(model_path, map_location='cpu', weights_only=False)
    tm   = ckpt.get('test_metrics', {})
    print(f"  HYP F1   : {tm.get('hyp_f1',   0):.3f}  (baseline 0.375)")
    print(f"  Macro F1 : {tm.get('macro_f1', 0):.3f}  (baseline 0.492)")
else:
    print('WARNING: ecgfm_stage2.pt not found — check output above for errors')